# Lab · Cross-entropy và khởi tạo xấu

Bài tập cho [`lesson.md`](lesson.md) (Module 6, Bài 2). Theo hai lab của khóa ("Logistic Regression and Bad Initialization Value" và "Logistic Regression Cross Entropy"): huấn luyện logistic regression từ một khởi tạo xấu với **MSE** rồi với **cross-entropy**, và so sánh.

Lưu ý: trang lab Cross Entropy gốc của khóa không có đường link tải file, nên phần cross-entropy ở đây được viết lại dựa trên video và lab MSE.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch", torch.__version__)

## Bài 1. Dữ liệu: nhãn 1 khi x > 0.2

Viết lớp `Data(Dataset)`: `self.x = torch.arange(-1, 1, 0.1).view(-1, 1)`; `self.y = torch.zeros(self.x.shape[0], 1)`; rồi đặt `self.y[self.x[:, 0] > 0.2] = 1`; `self.len`; `__getitem__`, `__len__` như thường lệ.
Tạo `data_set = Data()`. Lưu `n_pos` = số mẫu có nhãn 1. Tính tay trước: những `x` nào lớn hơn 0.2?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Đây là bài toán một đặc trưng, nhãn xác định bởi một ngưỡng. Mô hình sẽ phải tìm ra ngưỡng đó.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Gán bằng mặt nạ bool: `y[x[:, 0] > 0.2] = 1`. Đếm: `data_set.y.sum().item()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`x` chạy từ −1.0 đến 0.9; những giá trị lớn hơn 0.2 là 0.3, 0.4, …, 0.9.

</details>

In [ ]:
class Data(Dataset):
    pass  # TODO

data_set = None  # TODO
n_pos = None     # TODO
print(len(data_set), n_pos)

In [ ]:
assert len(data_set) == 20
assert n_pos == 7, "7 mẫu có x > 0.2"
print("✅ Bài 1 đúng (13 mẫu lớp 0, 7 mẫu lớp 1)")

## Bài 2. Mô hình và khởi tạo tham số

Viết lớp `logistic_regression(nn.Module)` (`self.linear = nn.Linear(n_inputs, 1)`; `forward` trả `torch.sigmoid(self.linear(x))`).
Viết hàm `make_model(w0, b0)` trả về mô hình `logistic_regression(1)` với `weight = w0`, `bias = b0` (đặt bằng `torch.no_grad()` và `copy_`).
Lưu `bad = make_model(-5.0, -10.0)` (khởi tạo "xấu") và `good = make_model(0.0, 0.0)`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Bạn muốn kiểm soát tham số khởi đầu để so sánh các cách huấn luyện công bằng.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`model.linear.weight.copy_(torch.tensor([[w0]]))`, `model.linear.bias.copy_(torch.tensor([b0]))`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

Bọc cả hai lệnh `copy_` trong `with torch.no_grad():`.

</details>

In [ ]:
class logistic_regression(nn.Module):
    pass  # TODO

def make_model(w0, b0):
    pass  # TODO

bad = None   # TODO
good = None  # TODO

In [ ]:
assert bad.linear.weight.item() == -5.0 and bad.linear.bias.item() == -10.0
assert good.linear.weight.item() == 0.0 and good.linear.bias.item() == 0.0
print("✅ Bài 2 đúng")

## Bài 3. Gradient lúc đầu: MSE so với cross-entropy

Với mô hình `bad` (chưa huấn luyện), tính loss trên **toàn bộ** `data_set.x`, `data_set.y` bằng hai cách rồi `backward()`:
- `nn.MSELoss()` → lưu `g_mse = abs(gradient của weight)`;
- `nn.BCELoss()` → lưu `g_bce = abs(gradient của weight)`.
Dùng một bản sao mô hình mới (gọi `make_model(-5., -10.)`) cho mỗi lần để gradient không dồn vào nhau.

**Đoán trước:** gradient nào lớn hơn, và lớn hơn bao nhiêu bậc?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Hai hàm loss khác nhau tạo ra hai bề mặt khác nhau quanh cùng điểm khởi đầu, nên cho hai gradient khác nhau.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`loss = criterion(model(data_set.x), data_set.y)`; `loss.backward()`; gradient ở `model.linear.weight.grad`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`g = abs(model.linear.weight.grad.item())`.

</details>

In [ ]:
g_mse = None  # TODO
g_bce = None  # TODO
print(g_mse, g_bce)

In [ ]:
assert g_mse < 1e-3, "MSE: gradient gần như bằng 0 (vùng phẳng)"
assert g_bce > 0.1, "cross-entropy: gradient vẫn đủ lớn để học"
assert g_bce / max(g_mse, 1e-12) > 1000
print("✅ Bài 3 đúng: chỗ này MSE không biết đi hướng nào, còn cross-entropy thì biết")

## Bài 4. Hàm huấn luyện chung

Viết `train(model, criterion, lr=2.0, epochs=100, batch_size=3)`:
- `DataLoader(data_set, batch_size=batch_size)` và `torch.optim.SGD(model.parameters(), lr=lr)`;
- vòng lặp huấn luyện chuẩn;
- trả về `(loss_first, loss_last, accuracy)`: loss trên **toàn bộ** dữ liệu **trước** khi huấn luyện, loss trên toàn bộ dữ liệu **sau** huấn luyện, và độ chính xác trên toàn bộ dữ liệu (nhãn dự đoán `p >= 0.5`, so với `data_set.y`).
Tính loss và accuracy dưới `torch.no_grad()`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Hàm này dùng lại cho mọi thí nghiệm: tham số `criterion` cho phép đổi loss mà không đổi mã.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Độ chính xác: `((model(x) >= 0.5).float() == y).float().mean().item()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`with torch.no_grad(): loss_first = criterion(model(data_set.x), data_set.y).item()` trước vòng lặp.

</details>

In [ ]:
def train(model, criterion, lr=2.0, epochs=100, batch_size=3):
    pass  # TODO

In [ ]:
_m = make_model(0.0, 0.0)
_a, _b, _acc = train(_m, nn.BCELoss(), lr=1.0, epochs=5)
assert _b < _a, "sau huấn luyện loss phải giảm"
assert 0.0 <= _acc <= 1.0
print("✅ Bài 4 đúng")

## Bài 5. MSE từ khởi tạo xấu

Huấn luyện mô hình khởi tạo xấu bằng **MSE**: `m_mse = make_model(-5., -10.)`, `r_mse = train(m_mse, nn.MSELoss())`.
Lưu `acc_mse = r_mse[2]` và `loss_change = abs(r_mse[0] - r_mse[1])`.
**Đoán trước:** độ chính xác bao nhiêu? (Nếu mô hình đoán toàn lớp 0 thì đúng bao nhiêu phần trăm?)

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Với gradient gần 0, các tham số gần như không đổi qua 100 epoch.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

Dùng hàm `train` bạn đã viết, truyền `nn.MSELoss()`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`r_mse = train(m_mse, nn.____())`.

</details>

In [ ]:
m_mse = None      # TODO
r_mse = None      # TODO
acc_mse = None    # TODO
loss_change = None  # TODO
print(r_mse)

In [ ]:
assert acc_mse < 0.7, "MSE từ khởi tạo xấu chỉ đạt khoảng 65%, tức là đoán toàn lớp 0 (13/20)"
assert loss_change < 0.01, "loss gần như không đổi: mô hình kẹt"
print("✅ Bài 5 đúng: mô hình kẹt")

## Bài 6. Cross-entropy từ cùng khởi tạo xấu

Lặp lại bài 5 nhưng với **`nn.BCELoss()`** (mô hình mới khởi tạo `-5, -10`). Lưu `r_bce`, `acc_bce` và `loss_drop = r_bce[0] - r_bce[1]`.
**Đoán trước** rồi so với bài 5: khác nhau ở đâu?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Mọi thứ giữ nguyên, chỉ đổi hàm loss.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`train(model, nn.BCELoss())`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`m_bce = make_model(____, ____)`.

</details>

In [ ]:
m_bce = None    # TODO
r_bce = None    # TODO
acc_bce = None  # TODO
loss_drop = None  # TODO
print(r_bce)

In [ ]:
assert acc_bce == 1.0, "cross-entropy thoát khỏi khởi tạo xấu và phân loại đúng cả 20 mẫu"
assert loss_drop > 2.0
print("✅ Bài 6 đúng: cùng dữ liệu, cùng khởi tạo, cùng optimizer; chỉ khác loss")

## Bài 7. MSE có luôn thất bại không?

Huấn luyện bằng **MSE** nhưng từ khởi tạo **tốt**: `make_model(0., 0.)`. Lưu `acc_mse_good`.
Câu hỏi: kết quả nói gì về nguyên nhân thất bại ở bài 5: do MSE nói chung hay do khởi tạo kết hợp với MSE?

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Hãy kiểm tra giả thuyết: nếu bắt đầu từ chỗ mà sigmoid chưa bão hòa, MSE có học được không?

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`make_model(0.0, 0.0)` rồi `train(..., nn.MSELoss())`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

Đọc phần tử cuối của kết quả `train`: độ chính xác.

</details>

In [ ]:
acc_mse_good = None  # TODO
print(acc_mse_good)

In [ ]:
assert acc_mse_good == 1.0
print("✅ Bài 7 đúng: MSE học được khi khởi tạo tốt; nó rất NHẠY với khởi tạo vì có vùng gradient biến mất")

## Bài 8. Cross-entropy bằng tay

`p = [0.9, 0.1, 0.8]` (xác suất dự đoán), `y = [1., 1., 0.]`.
- `bce_manual = -mean( y·log p + (1−y)·log(1−p) )` bằng tensor;
- `bce_torch = nn.BCELoss()(p, y)`;
- `small` = loss một mẫu khi `y = 1, p = 0.9`; `big` = loss một mẫu khi `y = 1, p = 0.1`; `ratio = big / small`.

<details><summary><b>Gợi ý 1</b> · hướng suy nghĩ</summary>

Với từng mẫu, chỉ một trong hai số hạng có tác dụng, tùy nhãn là 1 hay 0.

</details>

<details><summary><b>Gợi ý 2</b> · hàm / cách tiếp cận</summary>

`torch.log`. Loss một mẫu nhãn 1: `-torch.log(p)`.

</details>

<details><summary><b>Gợi ý 3</b> · gần như lời giải (thiếu vài chỗ)</summary>

`bce_manual = -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()`.

</details>

In [ ]:
p = torch.tensor([0.9, 0.1, 0.8])
y = torch.tensor([1.0, 1.0, 0.0])
bce_manual = None  # TODO
bce_torch = None   # TODO
small = None  # TODO
big = None    # TODO
ratio = None  # TODO
print(bce_manual, bce_torch, ratio)

In [ ]:
assert abs(bce_manual.item() - 1.3391) < 1e-3
assert torch.isclose(bce_manual, bce_torch)
assert abs(small - 0.1054) < 1e-3 and abs(big - 2.3026) < 1e-3
assert ratio > 20, "tự tin mà sai bị phạt nặng hơn hơn 20 lần"
print("✅ Bài 8 đúng")

## Xong rồi
Đối chiếu với `solution/`, rồi quay lại `lesson.md` nếu còn chỗ chưa chắc.